In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.payments
(
    payment_id INT,
    order_id INT,

    payment_date TIMESTAMP,

    payment_method STRING,
    payment_status STRING,

    amount DECIMAL(12,2),

    transaction_reference STRING,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("de_learning_workspace.bronze.payments")

silver_df = (
    bronze_df

    .dropDuplicates(["payment_id"])

    .withColumn(
        "payment_method",
        trim(col("payment_method"))
    )

    .withColumn(
        "payment_status",
        initcap(trim(col("payment_status")))
    )

    .filter(col("amount") >= 0)

    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("de_learning_workspace.silver.payments")
)